In [1]:
# ── 0. Install dependencies ──────────────────────────────────
# Run this cell once
!pip install openai datasets tqdm pandas -q

In [12]:
# ── OpenRouter client (one key for all models) ───────────────
from google.colab import userdata

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=userdata.get("OPENROUTER_API_KEY"),  # set in Colab secrets (key icon on left sidebar)
)

# ── Model config ─────────────────────────────────────────────
# Generator (small/cheap): GPT-4o-mini — fast, cheap, representative of lightweight agents
# Critic (large): GPT-4o — meaningfully stronger, used only in cross-model condition
# Critique classifier: GPT-4o-mini — cheap, consistent labeler
GENERATOR_MODEL  = "openai/gpt-4o-mini"
CRITIC_MODEL     = "openai/gpt-4o"
CLASSIFIER_MODEL = "openai/gpt-4o-mini"

# ── Reproducibility ──────────────────────────────────────────
SEED        = 42
NUM_SAMPLES = 250  # per benchmark — change to smaller int for testing

In [13]:
# ── 2. Prompt Templates ──────────────────────────────────────
# Standardized across all benchmarks — do NOT modify per benchmark.
# Benchmark-specific context is injected via the `task_description`
# and `answer_format` fields only.

def make_generate_prompt(task_description: str, question: str, answer_format: str) -> str:
    return f"""You are solving a {task_description}.

Question:
{question}

Instructions:
- Think step by step.
- {answer_format}
- Do not add any text after your final answer."""


def make_critique_prompt(task_description: str, question: str,
                          original_answer: str, critique_style: str = "targeted") -> str:
    if critique_style == "targeted":
        instruction = (
            "Identify specific errors in the answer above. "
            "For each error, explain what is wrong and why. "
            "If the answer is fully correct, say 'No errors found.'"
        )
    else:  # generic — used for ablation
        instruction = (
            "Review the answer above. "
            "Is it correct? If not, what could be improved?"
        )

    return f"""You are evaluating a solution to a {task_description}.

Question:
{question}

Answer to critique:
{original_answer}

Task:
{instruction}

Your critique:"""


def make_revise_prompt(task_description: str, question: str,
                        original_answer: str, critique: str, answer_format: str) -> str:
    return f"""You are solving a {task_description}.

Question:
{question}

Your previous answer:
{original_answer}

Critique of your answer:
{critique}

Instructions:
- Use the critique to fix any errors.
- If the critique says there are no errors, you may keep your answer.
- {answer_format}
- Do not add any text after your final answer.

Revised answer:"""


def make_classify_critique_prompt(critique: str) -> str:
    return f"""Classify the following critique into exactly one of these categories:
- accurate: the critique correctly identifies a real error
- vague: the critique is non-specific or unhelpful
- misleading: the critique incorrectly flags a correct answer or introduces wrong information

Critique:
{critique}

Respond with exactly one word: accurate, vague, or misleading."""

In [14]:
# ── 3. Core API Call ─────────────────────────────────────────
def call_model(prompt: str, model: str, max_tokens: int = 512,
               retries: int = 3) -> str:
    """Single-turn completion with retry logic."""
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=max_tokens,
                temperature=0.0,  # greedy — fully reproducible
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            wait = 2 ** attempt  # exponential backoff: 1s, 2s, 4s
            print(f"[WARN] Attempt {attempt+1} failed ({e}). Retrying in {wait}s...")
            time.sleep(wait)
    print(f"[ERROR] Model call failed after {retries} attempts.")
    return "[ERROR]"

In [15]:
# ── 4. Pipeline Runner ───────────────────────────────────────
def run_pipeline(
    samples: list[dict],          # list of {"id": ..., "question": ..., "answer": ...}
    task_description: str,
    answer_format: str,
    extract_answer_fn,            # fn(raw_output: str) -> str — benchmark specific
    evaluate_fn,                  # fn(pred: str, gold: str) -> bool — benchmark specific
    condition: str,               # "no_correction" | "same_model" | "cross_model"
    critique_style: str = "targeted",
) -> pd.DataFrame:
    """
    Runs one condition across all samples.
    Returns a DataFrame with one row per sample, all intermediate
    outputs logged for error transition analysis.
    """
    assert condition in ("no_correction", "same_model", "cross_model")

    records = []

    for s in tqdm(samples, desc=f"{condition} | {task_description}"):
        qid      = s["id"]
        question = s["question"]
        gold     = s["answer"]

        # ── GENERATE ────────────────────────────────────────
        gen_prompt  = make_generate_prompt(task_description, question, answer_format)
        raw_gen     = call_model(gen_prompt, GENERATOR_MODEL)
        pred_gen    = extract_answer_fn(raw_gen)
        correct_gen = evaluate_fn(pred_gen, gold)

        # ── CRITIQUE + REVISE (skipped for no_correction) ───
        critique = critique_class = raw_rev = pred_rev = correct_rev = None

        if condition in ("same_model", "cross_model"):
            critic_model = GENERATOR_MODEL if condition == "same_model" else CRITIC_MODEL

            crit_prompt = make_critique_prompt(
                task_description, question, raw_gen, critique_style
            )
            critique = call_model(crit_prompt, critic_model, max_tokens=256)

            rev_prompt = make_revise_prompt(
                task_description, question, raw_gen, critique, answer_format
            )
            raw_rev   = call_model(rev_prompt, GENERATOR_MODEL)
            pred_rev  = extract_answer_fn(raw_rev)
            correct_rev = evaluate_fn(pred_rev, gold)

            # Classify critique quality
            cls_prompt      = make_classify_critique_prompt(critique)
            critique_class  = call_model(cls_prompt, CLASSIFIER_MODEL, max_tokens=10)

        # ── Error transition label ───────────────────────────
        if condition == "no_correction":
            transition = None
        else:
            gen_lbl = "correct" if correct_gen else "wrong"
            rev_lbl = "correct" if correct_rev else "wrong"
            transition = f"{gen_lbl}→{rev_lbl}"

        records.append({
            "id":               qid,
            "question":         question,
            "gold":             gold,
            "condition":        condition,
            "critique_style":   critique_style,
            # Generation
            "raw_gen":          raw_gen,
            "pred_gen":         pred_gen,
            "correct_gen":      correct_gen,
            # Critique
            "critique":         critique,
            "critique_class":   critique_class,
            # Revision
            "raw_rev":          raw_rev,
            "pred_rev":         pred_rev,
            "correct_rev":      correct_rev,
            # Transition
            "transition":       transition,
        })

    return pd.DataFrame(records)

In [17]:
# ── 5. Summary Statistics ────────────────────────────────────
def summarize(df: pd.DataFrame) -> dict:
    """Compute accuracy and error transition counts from a results DataFrame."""
    out = {}
    out["n"] = len(df)
    out["gen_accuracy"] = df["correct_gen"].mean()

    if df["condition"].iloc[0] != "no_correction":
        out["rev_accuracy"]   = df["correct_rev"].mean()
        out["accuracy_delta"] = out["rev_accuracy"] - out["gen_accuracy"]
        out["transitions"]    = df["transition"].value_counts().to_dict()
        out["critique_quality"] = df["critique_class"].value_counts().to_dict()

    return out

In [18]:
# ── 6. Smoke Test ────────────────────────────────────────────
# Run this to verify your API key and pipeline work before
# plugging in real benchmark data.

def smoke_test():
    print("Running smoke test...")

    dummy_samples = [
        {"id": 0, "question": "What is 12 + 7?", "answer": "19"},
        {"id": 1, "question": "What is 100 - 43?", "answer": "57"},
    ]

    def extract(raw):
        # crude number extractor for smoke test only
        import re
        nums = re.findall(r'\b\d+\b', raw)
        return nums[-1] if nums else raw.strip()

    def evaluate(pred, gold):
        return pred.strip() == gold.strip()

    for cond in ("no_correction", "same_model", "cross_model"):
        df = run_pipeline(
            samples=dummy_samples,
            task_description="simple arithmetic problem",
            answer_format="End your response with 'Answer: <number>'",
            extract_answer_fn=extract,
            evaluate_fn=evaluate,
            condition=cond,
        )
        print(f"\n── {cond} ──")
        print(df[["id", "correct_gen", "transition", "critique_class"]])
        print("Summary:", summarize(df))

    print("\n✅ Smoke test passed — pipeline is working.")

smoke_test()

Running smoke test...


no_correction | simple arithmetic problem: 100%|██████████| 2/2 [00:07<00:00,  3.68s/it]



── no_correction ──
   id  correct_gen transition critique_class
0   0         True       None           None
1   1         True       None           None
Summary: {'n': 2, 'gen_accuracy': np.float64(1.0)}


same_model | simple arithmetic problem: 100%|██████████| 2/2 [00:19<00:00,  9.82s/it]



── same_model ──
   id  correct_gen       transition critique_class
0   0         True  correct→correct     misleading
1   1         True  correct→correct       accurate
Summary: {'n': 2, 'gen_accuracy': np.float64(1.0), 'rev_accuracy': np.float64(1.0), 'accuracy_delta': np.float64(0.0), 'transitions': {'correct→correct': 2}, 'critique_quality': {'misleading': 1, 'accurate': 1}}


cross_model | simple arithmetic problem: 100%|██████████| 2/2 [00:12<00:00,  6.31s/it]


── cross_model ──
   id  correct_gen       transition critique_class
0   0         True  correct→correct       accurate
1   1         True  correct→correct       accurate
Summary: {'n': 2, 'gen_accuracy': np.float64(1.0), 'rev_accuracy': np.float64(1.0), 'accuracy_delta': np.float64(0.0), 'transitions': {'correct→correct': 2}, 'critique_quality': {'accurate': 2}}

✅ Smoke test passed — pipeline is working.
